# 09 | M4b — 3D U-Net với Class-balanced Patch Sampling

**BraTS2020 · hồ sơ nghiên cứu M1–M6 · bản đọc/đối chiếu (clean review edition)**

> **Quan trọng:** Notebook này là bản trình bày có cấu trúc để đọc và review, **không phải Notebook tái chạy toàn bộ**. Mã thực nghiệm nguyên gốc đã được giữ trong `original_notebooks/09-original.ipynb`. Các phần quan trọng được trích nguyên văn vào `scripts/historical_cells/09/`. Không dùng **Run All** trên notebook gốc: một số cell có thể huấn luyện, tạo Dataset hoặc chạy test thật.

- Thí nghiệm: **M4b** — Class-balanced patch sampling.
- Nguồn: `09-m4b-patch-sampling-training (1).ipynb`; SHA256 của Notebook gốc: `aa558c157e6ace9c8acc48a9dddda76cfa2cdbce8fbff2ed8c488eaa40f5f388`.
- Git pin của pipeline dùng chung: `79a95a08f11694278114a89348cf8303a1e39d67` (branch `experiment/unet3d-optimization`).


## 1. Câu hỏi nghiên cứu & thay đổi được kiểm soát

Chỉ thay đổi cách lấy tâm patch, từ checkpoint M1 epoch 10.

**Tập dữ liệu cố định:** BraTS2020, 257 train / 55 internal validation / 56 reserved test, split seed 42. Đầu vào 4 MRI modalities (T1, T1ce, T2, FLAIR), đầu ra 4 logits (background và ba nhãn u đã remap). Train patch `96³`; full-volume inference ROI `96³`, overlap 0.25, Gaussian blending, CPU stitching.

**Riêng M2–M5:** tiếp tục từ M1 epoch 10; kết quả trình bày ở epoch 20.


## 2. Thiết kế kỹ thuật

- Chọn đều giữa các lớp u hiện diện, sau đó lấy ngẫu nhiên voxel thuộc lớp được chọn khi đi theo nhánh positive-center.
- Module `src/data/patches_class_balanced.py` đã xác minh SHA256 khớp nguyên bản đã chạy.

**Mã tham chiếu chính:**
- `src/data/patches_class_balanced.py (exact SHA256)`
- `src/data/patches.py`


## 3. Điểm khởi tạo / API cốt lõi (minh họa, không tự chạy)

Đoạn dưới giúp đọc kiến trúc và khác biệt chính, không tự khởi động training hoặc truy cập test:

```python
from src.data.patches_class_balanced import ClassBalancedRandomPatchDataset
# Xem source cell 04 và 15 để đối chiếu sampler với baseline.
```


## 4. Kết quả internal validation

**55 bệnh nhân · epoch 20 · macro-average Dice theo từng bệnh nhân.**

| WT | TC | ET | Mean Dice |
|---:|---:|---:|---:|
|0.857675132 | 0.733379235 | 0.661770548 | 0.750941638 |

- M4b Mean Dice = 0.750941638 (validation, epoch 20).

**Cách đọc:** Dice tốt hơn không đồng nghĩa có ý nghĩa thống kê; so sánh ghép cặp và Holm được lưu trong báo cáo phân tích M5.7 / M6.11.6.


## 5. Training và sao lưu đã được thực hiện như thế nào?

- Các cell huấn luyện gốc được triển khai trên Kaggle và lưu checkpoint/optimizer/AMP (tùy thí nghiệm), không lưu `.pt` trên GitHub.
- Sau khi chạy, các bản backup được xác minh bằng SHA256; kết quả tổng hợp từ những lần chạy đã hoàn tất được bảo quản trên Kaggle.
- Đừng coi đoạn code lưu trong `scripts/historical_cells/` là API thư viện chạy độc lập: chúng phụ thuộc biến trạng thái và mount của Kaggle, cần đưa vào module và kiểm thử trước khi tái sử dụng.
- Các lỗi và bước phục hồi lịch sử được giữ trong notebook gốc nhưng không nằm trong luồng đọc chính.


## 6. Chỉ mục mã nguồn gốc để audit

Những cell sau đã được trích **nguyên văn** vào thư mục `scripts/historical_cells/09/`. Cell index tính từ **0** trong file Notebook gốc.

| Nội dung | Cell index | File gốc đã trích |
|---|---:|---|
| Sampler exact source | 4 | `09/cell_04_restore_sampler.py` |
| Patch data loader | 15 | `09/cell_15_dataloader_setup.py` |
| Post-validation continuation | 37 | `09/cell_37_epoch16_to20_continuation.py` |
| Five-model merge | 43 | `09/cell_43_five_model_validation_merge.py` |
| Five-model statistics | 44 | `09/cell_44_five_model_statistics.py` |



## 7. Lưu ý tái lập và giới hạn

- **Đây không phải mã tái huấn luyện đã được kiểm thử sau refactor.** Muốn tái chạy phải xác minh phiên bản thư viện, split, SHA256 checkpoint nguồn, thứ tự thao tác và cơ chế backup.
- Những lần khôi phục phiên Kaggle có thể ảnh hưởng thứ tự RNG của DataLoader; checkpoint tại ranh giới epoch không cam kết bitwise identity.
- Không sử dụng kết quả từ test để điều chỉnh tham số sau khi đã khóa mô hình.
- Các mô hình khác kiến trúc (đặc biệt M6) phải được so sánh cả chi phí tính toán, không chỉ Dice.


In [ ]:
# Read-only: inspect fixed metadata from this review notebook.
# Safe to run locally; does not connect to Kaggle or load MRI.
experiment = 'M4b'
validation_mean_dice = 0.750941638
validation_cases = 55
print(experiment, "| validation patients:", validation_cases, "| Mean Dice:", validation_mean_dice)


## 9. Quy trình đưa lên GitHub

1. Giữ `original_notebooks` làm bằng chứng lịch sử; không thay thế/ghi đè bản gốc.
2. Chỉ chuyển module mới vào `src/` sau khi đối chiếu hàm/class nguyên bản và chạy smoke test.
3. Đồng bộ kết quả tổng hợp và checksum; **không commit `.pt`, `.nii`, credential, hoặc dữ liệu bệnh nhân chưa được duyệt công khai**.
4. Tạo PR riêng để review thay đổi trước khi merge vào `main`.
